In [66]:
import pandas as pd
import numpy as np

In [67]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer

In [68]:
df = pd.read_csv('/content/train.csv')[['Age','Fare','SibSp','Parch','Survived']]

In [69]:
df.dropna(inplace = True)

In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 714 entries, 0 to 890
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Age       714 non-null    float64
 1   Fare      714 non-null    float64
 2   SibSp     714 non-null    int64  
 3   Parch     714 non-null    int64  
 4   Survived  714 non-null    int64  
dtypes: float64(2), int64(3)
memory usage: 33.5 KB


In [71]:
df.head()

,Age,Fare,SibSp,Parch,Survived
0,22.0,7.2500,1,0,0
1,38.0,71.2833,1,0,1
2,26.0,7.9250,0,0,1
3,35.0,53.1000,1,0,1
4,35.0,8.0500,0,0,0


In [72]:
df['Family'] = df['SibSp'] + df['Parch']
df.head(5)

,Age,Fare,SibSp,Parch,Survived,Family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0


In [73]:
df.drop(columns = ['SibSp','Parch'],inplace = True,axis = 1)


In [74]:
df.head(5)

,Age,Fare,Survived,Family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [75]:
x = df.drop(columns=['Survived'])
y = df['Survived']


In [76]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

In [77]:
x_train.head()

,Age,Fare,Family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


## Without Binarization

In [78]:
clf = DecisionTreeClassifier()
clf.fit(x_train,y_train)

y_pred = clf.predict(x_test)

accuracy_score(y_test,y_pred)

0.6293706293706294

In [79]:
np.mean(cross_val_score(clf,x,y,cv=10,scoring='accuracy'))

np.float64(0.6415101721439751)

## With Binarization

In [80]:
from sklearn.preprocessing import Binarizer

In [81]:
trf = ColumnTransformer([
    ('bin', Binarizer(copy = False), ['Family'])
], remainder='passthrough')
x_train_trf = trf.fit_transform(x_train)
x_test_trf = trf.transform(x_test)

In [82]:
pd.DataFrame(x_train_trf,columns = ['Family','Age','Fare'])

,Family,Age,Fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [83]:
clf = DecisionTreeClassifier()
clf.fit(x_train_trf,y_train)

y_pred = clf.predict(x_test_trf)

accuracy_score(y_test,y_pred)

0.6153846153846154

In [84]:
np.mean(cross_val_score(clf,x,y,cv=10,scoring='accuracy'))

np.float64(0.6471244131455398)